# Supply Chain Twin: Google Colab notebook (data on Google Drive)

Generated by `scripts/build_notebook.py` (at commit `d209912`). The setup cell checks out the
code from GitHub (`REF`, default `main`) into `/content/SupplyTwinAI` and installs it, so Colab
runs exactly the pushed code; the checked-out commit is printed. To change the code, push to
GitHub (or set `REF` to another branch, tag or commit) and re-run the setup cell.

**Google Drive layout** (created on first run; change `DRIVE` in the setup cell to move it):

```
MyDrive/SupplyTwinAI/
  data/raw/provenance.json          dataset (pinned revision), about 110 MB
  data/raw/data/train.parquet
  data/raw/data/eval.parquet        locked final test: integrity-checked, never loaded
  runs/                             all run outputs (metrics, predictions, posteriors)
```

The dataset is read from Drive and copied to the Colab disk for fast reads. If it is not on
Drive yet, the pinned revision is downloaded once and saved there. You can also upload a
local `data/raw/` folder (with `provenance.json`) to that place yourself.

**How to use:** run *Setup*, *Helpers* and *Data* (Runtime → Run all is fine), then the steps
you need. Finished steps keep their outputs on Drive, so you can skip them later.
`FAST = True` (in the helpers cell) keeps every step to a few minutes; set `FAST = False` for
the full runs. If Colab restarts the session, re-run *Setup* and *Helpers* first.
Use a CPU runtime with many cores: the installed JAX is CPU-only, so a GPU is not used.

Rules that every step follows are in `CLAUDE.md` of the repository: rolling-origin
backtesting without leakage, point-in-time features, the official `eval` split is never read.
Sales are normalized units; reconstructed demand is never presented as observed.

## Setup

In [4]:
import os, sys, pathlib, shutil, subprocess
from google.colab import drive

REPO = "https://github.com/moghbaie/SupplyTwinAI.git"
REF = "main"  # branch, tag or commit to run
DRIVE = pathlib.Path("/content/drive/MyDrive/SupplyTwinAI")  # dataset and run outputs live here
ROOT = pathlib.Path("/content/SupplyTwinAI")  # code checkout (Colab disk, wiped per runtime)

def sh(cmd):
    """Run a shell command in ROOT; stop the cell if it fails."""
    subprocess.run(cmd, shell=True, check=True, cwd=ROOT)

drive.mount("/content/drive")
for d in (DRIVE / "data" / "raw", DRIVE / "runs", ROOT):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

# Check out the code (works in an empty folder and over leftovers of an older notebook;
# untracked files such as data/ and the runs link are kept, tracked files are overwritten).
if not (ROOT / ".git").exists():
    sh(f"git init -q && git remote add origin {REPO}")
sh(f"git fetch -q --depth 1 origin {REF} && git checkout -q --force FETCH_HEAD")
sh('git log -1 --format="code: commit %h (%cd) %s" --date=short')

if not os.path.islink("runs"):  # run outputs go straight to Drive
    if os.path.isdir("runs"):
        os.rmdir("runs")  # only succeeds if empty, so nothing is lost
    os.symlink(DRIVE / "runs", "runs")

# Install with all extras (no --force-reinstall: it would upgrade Colab's numpy and force a
# session restart). The import check runs in a fresh process, like the run cells below.
sh('pip install -q -e ".[dev,models,bayes]"')
sh('python -c "import sctwin.censoring, sctwin.demand_model; print(\'package import OK\')"')
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))  # make the package importable in this kernel too
print("code:", ROOT, "| data and runs:", DRIVE, "| CPU cores:", os.cpu_count())

Mounted at /content/drive
code: /content/SupplyTwinAI | data and runs: /content/drive/MyDrive/SupplyTwinAI | CPU cores: 12


## Helpers

In [5]:
import copy, glob, json
import pandas as pd
import yaml

os.chdir(ROOT)

FAST = True          # small, quick runs; set False for the full runs
CORES = os.cpu_count() or 2

def config_with(path, updates, name):
    """Copy a YAML config with nested updates applied; returns the new path."""
    cfg = yaml.safe_load(open(path, encoding="utf-8"))
    def merge(a, b):
        for k, v in b.items():
            if isinstance(v, dict) and isinstance(a.get(k), dict):
                merge(a[k], v)
            else:
                a[k] = v
    merge(cfg, copy.deepcopy(updates))
    out = f"configs/nb_{name}.yaml"
    yaml.safe_dump(cfg, open(out, "w", encoding="utf-8"), sort_keys=False)
    return out

def latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    if not files:
        raise FileNotFoundError(pattern)
    print("reading", files[-1])
    return json.load(open(files[-1], encoding="utf-8"))

def model_table(models):
    rows = {}
    for m, r in models.items():
        iv = r.get("intervals", {})
        rows[m] = {"mae": r.get("mae", r.get("mae_median")), "bias": r.get("bias"),
                   "wis": r.get("wis"), "cov50": iv.get("0.5", {}).get("coverage"),
                   "cov90": iv.get("0.9", {}).get("coverage")}
    return pd.DataFrame(rows).T.round(4)

## Run: tests (synthetic, no data needed)
The leakage tests run for both units, "day" and "hour". Warnings are errors, except
deprecation warnings: Colab's preinstalled libraries raise them inside each other (e.g.
matplotlib calling deprecated pyparsing), which is not this project's code. Local runs and CI
keep plain `-W error`.

In [6]:
!python -m pytest -W error -W "default::DeprecationWarning" -m "not data" -q -p no:cacheprovider

........................................................................ [ 64%]
........................................                                 [100%]
=============================== warnings summary ===============================
tests/test_baseline.py::test_runner_output_and_protocol[day]
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

tests/test_baseline.py::test_runner_output_and_protocol[day]
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
    parse = parser.parseString(pattern)

tests/test_baseline.py::test_runner_output_and_protocol[day]
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cach

## Data from Google Drive (run before any step)
Uses the dataset on Drive if present (checksums and pinned revision verified); otherwise
downloads the pinned revision once and saves it to Drive. Copies it to the Colab disk.

In [7]:
import json, yaml
from sctwin.data import verify_checksums

os.chdir(ROOT)
raw_drive = DRIVE / "data" / "raw"
pinned = yaml.safe_load(open("configs/small.yaml"))["dataset"]["revision"]
skip_cache = shutil.ignore_patterns(".cache")
if (raw_drive / "provenance.json").exists():
    print("dataset found on Drive:", raw_drive)
    shutil.copytree(raw_drive, "data/raw", dirs_exist_ok=True, ignore=skip_cache)
else:
    print("dataset not on Drive yet: downloading the pinned revision once")
    !python scripts/fetch_data.py
    shutil.copytree("data/raw", raw_drive, dirs_exist_ok=True, ignore=skip_cache)
    print("saved to Drive:", raw_drive)
prov = json.load(open("data/raw/provenance.json"))
assert prov["revision"] == pinned, f"Drive has revision {prov['revision']}, config pins {pinned}"
checks = verify_checksums("data/raw")
assert all(checks.values()), f"checksum mismatch: {checks}"
print("revision", pinned[:12], "checksums OK:", checks)

dataset found on Drive: /content/drive/MyDrive/SupplyTwinAI/data/raw
revision c374e85975c8 checksums OK: {'train': True, 'eval': True}


## Run: Step 1, audit and real-data tests
Re-runs the audit (it re-writes the identical split manifest) and the real-data tests,
including the real-data leakage checks.

In [ ]:
!python scripts/audit_data.py
!python -m pytest -m data -q -p no:cacheprovider

## Run: Step 3, baseline backtest (observed sales)
Rolling origin; train on feature times t <= T - L - H; predict (T + L, T + L + H].
Full day run: about 15 min on 12 cores and about 6 GB RAM. FAST uses 200 series and 2 folds.

In [ ]:
cfg3 = config_with("configs/baselines.yaml", {"lgbm": {"num_threads": CORES}}, "baselines")
smoke = "--smoke" if FAST else ""
!python scripts/run_baselines.py --unit day --config {cfg3} {smoke}
!python scripts/run_baselines.py --unit hour --config {cfg3} {smoke}

In [ ]:
for unit in ("day", "hour"):
    m = latest(f"runs/step3*/{unit}/*/metrics.json")
    print(unit, "origins", m["protocol"]["origins"][:5], "...")
    display(model_table(m["metrics"]["overall"]["models"]))
    display(pd.DataFrame({o: {k: v.get("wis", v.get("mae")) for k, v in f["models"].items()}
                          for o, f in m["metrics"]["folds"].items()}).T.round(4))

## Run: Step 4, censoring benchmark (proxy truth)
Demand-driven stockouts injected into fully in-stock days; calibration on the fit window only.
Full run: about 1.5 min for 5,000 series. FAST uses 300 series.

In [ ]:
smoke = "--smoke" if FAST else ""
!python scripts/run_censoring_benchmark.py {smoke}

In [ ]:
m = latest("runs/step4*/*/metrics.json")
print("calibrated injection:", m["calibrated_config"])
display(pd.DataFrame({"real": m["real_fit_window"]["stats"],
                      "injected": m["injected"]["stats"]}).round(3))
display(pd.DataFrame({"real": m["real_fit_window"]["position_ratios"],
                      "injected": m["injected"]["position_ratios"]}).round(3))
print("daily totals on stockout days")
display(model_table(m["benchmark"]["daily"]["overall"]["models"]))

## Run: Step 5, Bayesian model ladder (M0, M1, M2, M1d, M2d)
NumPyro NUTS. FAST: 4 series and short chains (a few minutes). Full: 20 series, 4 chains x
(1000 + 1000) draws; chains run in parallel, one per CPU core (up to 4), so it takes hours on a
2-core runtime. `--part real` and `--part recovery` split the run in two.
M3 = M2 with `--n-series 100 --models M2`. M1d/M2d are exploratory (added after the plan).

In [8]:
chains = min(4, CORES)
cfg5 = config_with("configs/bayes.yaml", {"mcmc": {"num_chains": chains}}, "bayes")
smoke = "--smoke" if FAST else ""
!python scripts/run_bayes.py --config {cfg5} --part all {smoke}

fit: 4 series x 76 days; benchmark: 31 days, 12 with a stockout
[M0] fitting ...
[M0] 72.0 s, divergences=0, max R-hat=1.032, min ESS=36
/usr/local/lib/python3.13/dist-packages/arviz_stats/loo/loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.50 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
[M1] fitting ...
[M1] 56.2 s, divergences=0, max R-hat=1.130, min ESS=-19847
/usr/local/lib/python3.13/dist-packages/arviz_stats/loo/loo_helper.py:1169: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.50 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterio

In [9]:
m = latest("runs/step5*/*/metrics_all.json")
real = m["real"]
keys = ("runtime_s", "divergences", "max_rhat", "min_ess")
display(pd.DataFrame({k: {x: v["diagnostics"][x] for x in keys}
                      for k, v in real["models"].items()}).T)
print("daily totals on stockout days (proxy truth); baselines are point estimates")
display(model_table(real["daily"]["overall"]["models"]))
print("PSIS-LOO on in-stock hours:", real["loo_compare"])
for name, r in m["recovery"]["models"].items():
    print("recovery", name, r["mean_window_demand"])
    display(pd.DataFrame(r["scalars"]).T)

reading runs/step5_smoke/6c6463c48440/metrics_all.json


,runtime_s,divergences,max_rhat,min_ess
M0,72.0,0.0,1.032203,36.087999
M1,56.2,0.0,1.129879,-19846.540420
M2,151.3,0.0,1.055129,20.557950
M1d,57.2,0.0,1.087697,34.157846
M2d,171.4,0.0,1.079042,24.208006


daily totals on stockout days (proxy truth); baselines are point estimates


,mae,bias,wis,cov50,cov90
M0,0.1442,-0.1052,0.0674,0.3333,1.0000
M1,0.1447,0.1061,0.0824,0.5833,0.9167
M1d,0.1125,0.0647,0.0644,0.6667,1.0000
M2,0.1361,0.0961,0.0757,0.5833,1.0000
M2d,0.1200,0.0662,0.0689,0.7500,1.0000
hour_mean,0.1486,-0.1024,NaN,NaN,NaN
observed,0.3203,-0.3203,NaN,NaN,NaN
profile,0.1194,-0.0393,NaN,NaN,NaN
profile_pre,0.1412,-0.0045,NaN,NaN,NaN


PSIS-LOO on in-stock hours: {'best': 'M0', 'models': {'M0': {'elpd_diff': 0.0, 'se_diff': 0.0}, 'M1': {'elpd_diff': -7.018611578722849, 'se_diff': 4.912992790569302}, 'M2': {'elpd_diff': -11.497126595117932, 'se_diff': 4.516545777075596}, 'M1d': {'elpd_diff': -8.595985215841816, 'se_diff': 4.5625773480359975}, 'M2d': {'elpd_diff': -10.156376099924348, 'se_diff': 4.7836066539775794}}}
recovery M0 {'mean': 2.094112141255019, 'q05': 1.9963455101614411, 'q95': 2.1811269405928657, 'covered': False}


,true,mean,q05,q95,covered
alpha,-2.0,-2.23869,-2.653716,-1.805943,True
sigma_a,0.8,0.515626,0.20917,1.069224,True
tau_g,0.3,0.25312,0.183861,0.328582,True
sigma_u,0.35,0.333479,0.279929,0.381007,True
b_disc,0.8,0.320731,-0.899347,1.542978,True
b_act,0.3,0.175159,-0.090125,0.431291,True
sigma_y,0.9,0.908044,0.885789,0.932599,True
k0,0.5,0.338589,-0.123893,0.82975,True
k1,1.0,1.113654,1.026607,1.202658,False


... (4 outputs left out)

recovery M1d {'mean': 2.2413766870528993, 'q05': 2.1372065665249074, 'q95': 2.366263701498575, 'covered': True}


,true,mean,q05,q95,covered
alpha,-2.0,-2.214841,-2.59522,-1.879419,True
sigma_a,0.8,0.487094,0.209889,0.940376,True
tau_g,0.3,0.247312,0.181207,0.327816,True
sigma_u,0.35,0.348847,0.291527,0.402524,True
b_disc,0.8,0.108747,-1.162436,1.272188,True
b_act,0.3,0.176116,-0.083832,0.406971,True
sigma_y,0.9,0.896347,0.874453,0.920518,True
k0,0.5,0.306999,-0.101793,0.661333,True
k1,1.0,1.080356,0.967625,1.18679,True


recovery M2d {'mean': 2.2313896170233716, 'q05': 2.122702042313341, 'q95': 2.331514369891485, 'covered': True}


,true,mean,q05,q95,covered
alpha,-2.0,-2.214407,-2.703487,-1.757019,True
sigma_a,0.8,0.541366,0.238384,1.048986,True
tau_g,0.3,0.252436,0.188356,0.32554,True
sigma_u,0.35,0.327395,0.264253,0.390709,True
b_disc,0.8,0.040271,-1.355151,1.590238,True
b_act,0.3,0.198351,-0.105456,0.516767,True
sigma_y,0.9,0.896609,0.875919,0.918729,True
k0,0.5,0.311659,-0.227462,0.843915,True
k1,1.0,1.083677,0.99335,1.171021,True
rho,0.5,0.354921,0.154665,0.524585,True


## Run: Step 5, simulation-based calibration of M1
Ranks of the true parameters among posterior draws should be uniform (chi-square p-values not
small after Holm). Full: 200 replications (about 40 min). FAST: 20.

In [10]:
reps = 20 if FAST else 200
!python scripts/run_sbc.py --reps {reps}
s = latest("runs/sbc/*/ranks.json")
display(pd.DataFrame(s["scalars"]).T)
print("min Holm-adjusted p:", s["min_p_holm"])
print("divergent replications:", s["divergent_reps"])

rep 10/20  427 s
rep 20/20  687 s
  alpha    chi2 p=0.276 mean rank=0.43 [4, 1, 2, 1, 5, 3, 1, 0, 1, 2]
  sigma_a  chi2 p=0.067 mean rank=0.39 [5, 5, 0, 0, 3, 1, 1, 3, 1, 1]
  tau_g    chi2 p=0.350 mean rank=0.38 [4, 4, 4, 1, 0, 1, 2, 1, 2, 1]
  sigma_u  chi2 p=0.001 mean rank=0.50 [0, 1, 2, 4, 3, 1, 8, 1, 0, 0]
  b_disc   chi2 p=0.911 mean rank=0.47 [1, 3, 3, 3, 1, 2, 1, 3, 2, 1]
  b_act    chi2 p=0.035 mean rank=0.51 [0, 2, 1, 1, 6, 4, 4, 1, 0, 1]
  sigma_y  chi2 p=0.911 mean rank=0.41 [2, 3, 3, 3, 3, 2, 1, 1, 1, 1]
  k0       chi2 p=0.534 mean rank=0.46 [2, 2, 2, 3, 3, 0, 3, 4, 1, 0]
  k1       chi2 p=0.213 mean rank=0.62 [2, 0, 2, 3, 0, 1, 1, 5, 2, 4]
min Holm-adjusted p = 0.009; written to runs/sbc/20a4d182c47b
reading runs/sbc/20a4d182c47b/ranks.json


,hist,chi2_p,mean_rank_frac
alpha,"[4, 1, 2, 1, 5, 3, 1, 0, 1, 2]",0.275709,0.4345
sigma_a,"[5, 5, 0, 0, 3, 1, 1, 3, 1, 1]",0.066882,0.3885
tau_g,"[4, 4, 4, 1, 0, 1, 2, 1, 2, 1]",0.350485,0.384
sigma_u,"[0, 1, 2, 4, 3, 1, 8, 1, 0, 0]",0.000954,0.501
b_disc,"[1, 3, 3, 3, 1, 2, 1, 3, 2, 1]",0.911413,0.473
b_act,"[0, 2, 1, 1, 6, 4, 4, 1, 0, 1]",0.035174,0.512
sigma_y,"[2, 3, 3, 3, 3, 2, 1, 1, 1, 1]",0.911413,0.4055
k0,"[2, 2, 2, 3, 3, 0, 3, 4, 1, 0]",0.534146,0.457
k1,"[2, 0, 2, 3, 0, 1, 1, 5, 2, 4]",0.213309,0.6235


min Holm-adjusted p: 0.008584804949451713
divergent replications: 3
